In [6]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import kagglehub
from kagglehub import KaggleDatasetAdapter
from gensim.models import Word2Vec

In [7]:
file_path = "positions.csv"
data = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "nikitricky/chess-positions",
  file_path
)

Using Colab cache for faster access to the 'chess-positions' dataset.


/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [8]:
moves_corpus = [[move] for move in data['playing']]
model = Word2Vec(sentences=moves_corpus, vector_size=100, window=5, min_count=1, workers=4)

In [9]:
print("First 5 records:", data.head())
#print(np.array(data.iloc[0][0]))


First 5 records:                                                  fen playing  score  mate  \
0  rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...    e2e4  -35.0   NaN   
1  rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...    e7e6   36.0   NaN   
2  rnbqkbnr/pppp1ppp/4p3/8/3PP3/8/PPP2PPP/RNBQKBN...    d2d4  -27.0   NaN   
3  rnbqkbnr/p1pp1ppp/1p2p3/8/3PP3/8/PPP2PPP/RNBQK...    b7b6   81.0   NaN   
4  rnbqkbnr/p1pp1ppp/1p2p3/8/3PP3/P7/1PP2PPP/RNBQ...    a2a3  -65.0   NaN   

   depth   game_id        date      time  white    black white_result  \
0     20  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
1     22  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
2     24  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
3     20  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
4     22  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   

  black_result white_elo black_elo                           opening  \
0        

In [10]:
vector = model.wv['e2e4']
print(vector)
print(model.wv.most_similar('e2e4', topn=5))

[ 9.4563962e-05  3.0773198e-03 -6.8126451e-03 -1.3754654e-03
  7.6685809e-03  7.3464094e-03 -3.6732971e-03  2.6427018e-03
 -8.3171297e-03  6.2054861e-03 -4.6373224e-03 -3.1641065e-03
  9.3113566e-03  8.7338570e-04  7.4907029e-03 -6.0740625e-03
  5.1605068e-03  9.9228229e-03 -8.4573915e-03 -5.1356913e-03
 -7.0648370e-03 -4.8626517e-03 -3.7785638e-03 -8.5361991e-03
  7.9556061e-03 -4.8439382e-03  8.4236134e-03  5.2625705e-03
 -6.5500261e-03  3.9578713e-03  5.4701497e-03 -7.4265362e-03
 -7.4057197e-03 -2.4752307e-03 -8.6257253e-03 -1.5815723e-03
 -4.0343284e-04  3.2996845e-03  1.4418805e-03 -8.8142155e-04
 -5.5940580e-03  1.7303658e-03 -8.9737179e-04  6.7936908e-03
  3.9735902e-03  4.5294715e-03  1.4343059e-03 -2.6998555e-03
 -4.3668128e-03 -1.0320747e-03  1.4370275e-03 -2.6460087e-03
 -7.0737829e-03 -7.8053069e-03 -9.1217868e-03 -5.9351693e-03
 -1.8474245e-03 -4.3238713e-03 -6.4606704e-03 -3.7173224e-03
  4.2891586e-03 -3.7390434e-03  8.3781751e-03  1.5339935e-03
 -7.2423196e-03  9.43379

In [11]:
import numpy as np

def get_embedding(word, w2v_model, vector_size=100):
    """Returns the Word2Vec embedding for a word, or a zero vector if not found."""
    if word in w2v_model.wv:
        return w2v_model.wv[word]
    else:
        return np.zeros(vector_size)

### Generating Embeddings for Categorical Columns

Now, let's generate Word2Vec embeddings for `white`, `black`, `opening`, `time_control`, and `termination` columns. We will train a separate Word2Vec model for each of these columns. The existing `model` is already trained on the `playing` column.

In [12]:
# Columns to embed with Word2Vec
columns_to_embed = ['white', 'black', 'opening', 'time_control', 'termination']

# Create new DataFrame to store embeddings
data_embeddings = pd.DataFrame()

# Process 'playing' column using the existing model
playing_embeddings = np.array([get_embedding(move, model, model.vector_size) for move in data['playing']])
playing_embedding_df = pd.DataFrame(playing_embeddings, columns=[f'playing_emb_{i}' for i in range(model.vector_size)])
data_embeddings = pd.concat([data_embeddings, playing_embedding_df], axis=1)

# Process other columns
for col in columns_to_embed:
    # Create a corpus for the current column
    col_corpus = [[str(item)] for item in data[col].fillna('')] # Handle potential NaN values

    # Train a new Word2Vec model for the column
    col_w2v_model = Word2Vec(sentences=col_corpus, vector_size=100, window=5, min_count=1, workers=4)

    # Generate embeddings
    col_embeddings = np.array([get_embedding(str(item), col_w2v_model, 100) for item in data[col].fillna('')])

    # Create a DataFrame for the current column's embeddings
    col_embedding_df = pd.DataFrame(col_embeddings, columns=[f'{col}_emb_{i}' for i in range(100)])

    # Concatenate to the main embeddings DataFrame
    data_embeddings = pd.concat([data_embeddings, col_embedding_df], axis=1)

display(data_embeddings.head())
print(f"Shape of data_embeddings: {data_embeddings.shape}")

,playing_emb_0,playing_emb_1,playing_emb_2,playing_emb_3,playing_emb_4,playing_emb_5,playing_emb_6,playing_emb_7,playing_emb_8,playing_emb_9,...,termination_emb_90,termination_emb_91,termination_emb_92,termination_emb_93,termination_emb_94,termination_emb_95,termination_emb_96,termination_emb_97,termination_emb_98,termination_emb_99
0,0.000095,0.003077,-0.006813,-0.001375,0.007669,0.007346,-0.003673,0.002643,-0.008317,0.006205,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
1,-0.009579,0.008943,0.004165,0.009235,0.006644,0.002925,0.009804,-0.004425,-0.006803,0.004227,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
2,-0.008243,0.009299,-0.000198,-0.001967,0.004604,-0.004095,0.002743,0.006940,0.006065,-0.007511,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
3,-0.006964,-0.002459,-0.008023,0.007501,0.006127,0.005258,0.008378,-0.000697,-0.009313,0.009116,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
4,0.001333,0.006541,0.009985,0.009062,-0.008015,0.006491,-0.005715,-0.000972,0.000483,0.006582,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393


Shape of data_embeddings: (1091078, 600)


### Prepare Data for Machine Learning Model

Now, we will combine these embeddings with other numerical features from the original dataset. We'll drop the original string columns that have been embedded and the `opening_encoded` column, which is now superseded by the `opening` Word2Vec embeddings. We will also define `white_result` as our target variable `y`.

In [13]:
# Select numerical columns from the original data (excluding `score` which has NaNs and `mate`)
numerical_cols = data.select_dtypes(include=np.number).columns.tolist()
# Exclude 'score' and 'mate' for now as they have many NaNs or mixed types
# `white_elo` and `black_elo` are already numbers.
# `depth` is a number.

# Filter out 'opening_encoded' if it exists, and 'mate' and 'score' for simplicity due to NaNs
numerical_features = data[['depth', 'white_elo', 'black_elo']].copy()

# Combine numerical features with embeddings
X = pd.concat([numerical_features, data_embeddings], axis=1);

# Handle remaining NaN values in X (e.g., fill with 0 or mean)
X = X.fillna(0) # Filling with 0 as a simple strategy

# Define the target variable (white_result, converted to numeric)
# Convert '1/2' to '0' and then to int for binary classification
y = data['white_result'].replace({'1/2': '0'}).astype(int)

print(f"Shape of feature matrix X: {X.shape}")
print(f"Shape of target vector y: {y.shape}")
display(X.head())

Shape of feature matrix X: (1091078, 603)
Shape of target vector y: (1091078,)


,depth,white_elo,black_elo,playing_emb_0,playing_emb_1,playing_emb_2,playing_emb_3,playing_emb_4,playing_emb_5,playing_emb_6,...,termination_emb_90,termination_emb_91,termination_emb_92,termination_emb_93,termination_emb_94,termination_emb_95,termination_emb_96,termination_emb_97,termination_emb_98,termination_emb_99
0,20,1639,1403,0.000095,0.003077,-0.006813,-0.001375,0.007669,0.007346,-0.003673,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
1,22,1639,1403,-0.009579,0.008943,0.004165,0.009235,0.006644,0.002925,0.009804,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
2,24,1639,1403,-0.008243,0.009299,-0.000198,-0.001967,0.004604,-0.004095,0.002743,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
3,20,1639,1403,-0.006964,-0.002459,-0.008023,0.007501,0.006127,0.005258,0.008378,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393
4,22,1639,1403,0.001333,0.006541,0.009985,0.009062,-0.008015,0.006491,-0.005715,...,0.001631,0.00019,0.003474,0.000218,0.009619,0.005061,-0.008917,-0.007042,0.000901,0.006393


### Train and Evaluate a Simple Model

Finally, we will split the data into training and testing sets and train a `RandomForestClassifier` to predict `white_result`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train a RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
classifier.fit(X_train, y_train);

# Make predictions on the test set
y_pred = classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")